# Задание 4: Создание и заполнение БД PostgreSQL — Логи образовательного портала

In [1]:
import os
import csv
import psycopg2
import pandas as pd
from psycopg2.extras import DictCursor

POSTGRESQL_HOST = os.environ.get('POSTGRESQL_HOST', 'localhost')
POSTGRESQL_USER = os.environ.get('POSTGRESQL_USER', 'postgres')
POSTGRESQL_PASSWORD = os.environ.get('POSTGRESQL_PASSWORD', 'postgres')
DATASET_PATH = 'datasets/aggrigation_logs_per_week.csv'


### Создание тестовой базы данных и подключение

In [2]:
# Создаём БД test (если не существует)
conn_default = psycopg2.connect(
    dbname='postgres',
    user=POSTGRESQL_USER,
    password=POSTGRESQL_PASSWORD,
    host=POSTGRESQL_HOST
)
conn_default.autocommit = True
with conn_default.cursor() as cur:
    cur.execute("SELECT 1 FROM pg_database WHERE datname='test'")
    if not cur.fetchone():
        cur.execute('CREATE DATABASE test')
        print("БД 'test' создана")
    else:
        print("БД 'test' уже существует")
conn_default.close()


БД 'test' уже существует


In [3]:
# Подключаемся к test
conn = psycopg2.connect(
    dbname='test',
    user=POSTGRESQL_USER,
    password=POSTGRESQL_PASSWORD,
    host=POSTGRESQL_HOST
)
conn.autocommit = True
print("Подключение к БД test успешно")


Подключение к БД test успешно


### Просмотр заголовков CSV

In [4]:
with open(DATASET_PATH) as file:
    reader = csv.reader(file)
    headers = next(reader)
    first_row = next(reader)

print('Заголовки:', headers)
print('Первая строка:', first_row)


Заголовки: ['courseid', 'userid', 'num_week', 's_all', 's_all_avg', 's_course_viewed', 's_course_viewed_avg', 's_q_attempt_viewed', 's_q_attempt_viewed_avg', 's_a_course_module_viewed', 's_a_course_module_viewed_avg', 's_a_submission_status_viewed', 's_a_submission_status_viewed_avg', 'NameR_Level', 'Name_vAtt', 'Depart', 'Name_OsnO', 'Name_FormOPril', 'LevelEd', 'Num_Sem', 'Kurs', 'Date_vAtt']
Первая строка: ['71262', '34527', '6', '9', '9', '4', '4', '0', '0', '0', '0', '0', '0', '3', 'Экзамен', '22', '1', '1', '1', '2', '2', '18.06.2022']


### Создание таблицы logs и загрузка данных

In [5]:
with conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS logs;")
    cur.execute("""
        CREATE TABLE logs (
            courseid                         varchar(10),
            userid                           varchar(10),
            num_week                         smallint,
            s_all                            smallint,
            s_all_avg                        varchar(20),
            s_course_viewed                  smallint,
            s_course_viewed_avg              varchar(20),
            s_q_attempt_viewed               smallint,
            s_q_attempt_viewed_avg           varchar(20),
            s_a_course_module_viewed         smallint,
            s_a_course_module_viewed_avg     varchar(20),
            s_a_submission_status_viewed     smallint,
            s_a_submission_status_viewed_avg varchar(20),
            NameR_Level                      varchar(20),
            Name_vAtt                        varchar(20),
            Depart                           varchar(5),
            Name_OsnO                        smallint,
            Name_FormOPril                   smallint,
            LevelEd                          smallint,
            Num_Sem                          smallint,
            Kurs                             smallint,
            Date_vAtt                        varchar(30)
        );
    """)
    print("Таблица logs создана")


Таблица logs создана


In [6]:
# Загружаем данные из CSV
with conn.cursor() as cur:
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        next(f)  # пропускаем заголовок
        cur.copy_expert(
            """COPY logs FROM STDIN WITH (FORMAT CSV, DELIMITER ',', NULL '')""",
            f
        )
    
with conn.cursor() as cur:
    cur.execute('SELECT COUNT(*) FROM logs')
    count = cur.fetchone()[0]
print(f"Загружено записей: {count:,}")


Загружено записей: 414,528


In [7]:
# Первые 10 записей
with conn.cursor() as cur:
    cur.execute('SELECT * FROM logs LIMIT 10')
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]

df = pd.DataFrame(rows, columns=cols)
df


,courseid,userid,num_week,s_all,s_all_avg,s_course_viewed,s_course_viewed_avg,s_q_attempt_viewed,s_q_attempt_viewed_avg,s_a_course_module_viewed,...,s_a_submission_status_viewed_avg,namer_level,name_vatt,depart,name_osno,name_formopril,leveled,num_sem,kurs,date_vatt
0,71262,34527,6,9,9,4,4,0,0,0,...,0,3,Экзамен,22,1,1,1,2,2,18.06.2022
1,71262,34527,7,0,"4,5",0,2,0,0,0,...,0,3,Экзамен,22,1,1,1,2,2,18.06.2022
2,71262,34527,8,0,3,0,"1,3333",0,0,0,...,0,3,Экзамен,22,1,1,1,2,2,18.06.2022
3,71262,34527,9,0,"2,25",0,1,0,0,0,...,0,3,Экзамен,22,1,1,1,2,2,18.06.2022
4,71262,34527,10,0,"1,8",0,"0,8",0,0,0,...,0,3,Экзамен,22,1,1,1,2,2,18.06.2022
5,71262,34527,11,1,"1,6667",1,"0,8333",0,0,0,...,0,3,Экзамен,22,1,1,1,2,2,18.06.2022
6,71262,34527,12,0,"1,4286",0,"0,7143",0,0,0,...,0,3,Экзамен,22,1,1,1,2,2,18.06.2022
7,71262,34527,13,0,"1,25",0,"0,625",0,0,0,...,0,3,Экзамен,22,1,1,1,2,2,18.06.2022
8,71262,34527,14,0,"1,1111",0,"0,5556",0,0,0,...,0,3,Экзамен,22,1,1,1,2,2,18.06.2022
9,71262,34527,15,1,"1,1",1,"0,6",0,0,0,...,0,3,Экзамен,22,1,1,1,2,2,18.06.2022


## Задание 1: Замена запятой на точку в вещественных полях

In [8]:
# Поля с вещественными значениями (avg-колонки)
float_columns = [
    's_all_avg',
    's_course_viewed_avg',
    's_q_attempt_viewed_avg',
    's_a_course_module_viewed_avg',
    's_a_submission_status_viewed_avg'
]

with conn.cursor() as cur:
    for col in float_columns:
        cur.execute(f"UPDATE logs SET {col} = REPLACE({col}, ',', '.') WHERE {col} LIKE '%,%';")
    conn.commit() if not conn.autocommit else None

print("Замена запятых на точки выполнена")

# Проверяем результат
with conn.cursor() as cur:
    cur.execute('SELECT s_all_avg, s_course_viewed_avg FROM logs LIMIT 10')
    rows = cur.fetchall()
for row in rows:
    print(row)


Замена запятых на точки выполнена
('9', '4')
('6', '3')
('13', '5')
('3', '1.3333')
('1', '0.5455')
('1', '0.5714')
('1', '0.6')
('8', '4')
('4', '2')
('2', '1')


## Задание 2: Количество кафедр

In [9]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(DISTINCT depart) AS num_departments FROM logs;")
    row = cur.fetchone()
print(f"Количество уникальных кафедр: {row[0]}")


Количество уникальных кафедр: 43


## Задание 3: Количество электронных курсов по каждой кафедре

In [10]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            depart,
            COUNT(DISTINCT courseid) AS num_courses
        FROM logs
        GROUP BY depart
        ORDER BY num_courses DESC;
    """)
    rows = cur.fetchall()

print(f"{'Кафедра':<10} {'Курсов':>10}")
print("-"*22)
for depart, num in rows:
    print(f"{depart:<10} {num:>10}")

top = rows[0]
print(f"\nКафедра с наибольшим количеством курсов: {top[0]} ({top[1]} курсов)")


Кафедра        Курсов
----------------------
11                 53
19                 42
23                 42
8                  40
15                 36
4                  35
9                  35
21                 33
14                 33
3                  33
41                 32
20                 32
31                 30
7                  29
26                 28
16                 28
18                 28
6                  25
24                 23
5                  22
32                 21
43                 20
10                 20
17                 20
22                 19
30                 19
29                 19
1                  19
37                 18
38                 18
27                 17
12                 16
35                 16
40                 16
42                 16
2                  16
33                 15
28                 14
25                 11
36                  5
39                  4
13                  3
34                  2

Кафедра 

## Задание 4: Создание таблицы departments со справочником кафедр

In [11]:
departments_data = [
    (1,'АиИИ'),(2,'АСУ'),(3,'АЭПиМ'),(4,'БИиИТ'),(5,'ВИ'),
    (6,'ВТиП'),(7,'ГМДиОПИ'),(8,'ГМиТТК'),(9,'ГМУиУП'),(10,'Дизайна'),
    (11,'ДиСО'),(12,'ИиИБ'),(13,'ИТМ'),(14,'ЛиП'),(15,'ЛиУТС'),
    (16,'ЛПиМ'),(17,'Менеджм.'),(18,'МиТОДиМ'),(19,'МиХТ'),(20,'ПиСЗ'),
    (21,'ПиЭММО'),(22,'ПМиИ'),(23,'ПОиД'),(24,'Психол.'),(25,'ПЭиБЖД'),
    (26,'РМПИ'),(27,'РЯОЯиМК'),(28,'СРиППО'),(29,'СС'),(30,'ТиЭС'),
    (31,'ТОМ'),(32,'ТССА'),(33,'УиИС'),(34,'УСиБА'),(35,'Физики'),
    (36,'Физкульт.'),(37,'Химии'),(38,'ХОМ'),(39,'ЦДОМ'),(40,'ЭиМЭ'),
    (41,'Эконом.'),(42,'ЭПП'),(43,'ЯиЛ')
]

with conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS departments;")
    cur.execute("""
        CREATE TABLE departments (
            id   smallint PRIMARY KEY,
            name varchar(50) NOT NULL
        );
    """)
    cur.executemany("INSERT INTO departments VALUES (%s, %s)", departments_data)
    cur.execute("SELECT COUNT(*) FROM departments")
    count = cur.fetchone()[0]

print(f"Таблица departments создана и заполнена: {count} записей")

with conn.cursor() as cur:
    cur.execute("SELECT * FROM departments LIMIT 5")
    for row in cur.fetchall():
        print(row)


Таблица departments создана и заполнена: 43 записей
(1, 'АиИИ')
(2, 'АСУ')
(3, 'АЭПиМ')
(4, 'БИиИТ')
(5, 'ВИ')


## Задание 5: Количество курсов по кафедрам (с названиями кафедр)

In [12]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            d.name      AS department_name,
            COUNT(DISTINCT l.courseid) AS num_courses
        FROM logs l
        JOIN departments d ON l.depart::smallint = d.id
        GROUP BY d.name
        ORDER BY num_courses DESC;
    """)
    rows = cur.fetchall()

print(f"{'Кафедра':<20} {'Курсов':>10}")
print("-"*32)
for name, num in rows:
    print(f"{name:<20} {num:>10}")

print(f"\nБольше всего курсов: {rows[0][0]} ({rows[0][1]})")


Кафедра                  Курсов
--------------------------------
ДиСО                         53
МиХТ                         42
ПОиД                         42
ГМиТТК                       40
ЛиУТС                        36
БИиИТ                        35
ГМУиУП                       35
АЭПиМ                        33
ПиЭММО                       33
ЛиП                          33
Эконом.                      32
ПиСЗ                         32
ТОМ                          30
ГМДиОПИ                      29
РМПИ                         28
ЛПиМ                         28
МиТОДиМ                      28
ВТиП                         25
Психол.                      23
ВИ                           22
ТССА                         21
Менеджм.                     20
ЯиЛ                          20
Дизайна                      20
ПМиИ                         19
АиИИ                         19
СС                           19
ТиЭС                         19
ХОМ                          18
Химии  

## Задание 6: Курсы, за которыми закреплено несколько кафедр

In [13]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT courseid, COUNT(DISTINCT depart) AS dept_count
        FROM logs
        GROUP BY courseid
        HAVING COUNT(DISTINCT depart) > 1
        ORDER BY dept_count DESC;
    """)
    rows = cur.fetchall()

if rows:
    print(f"Курсов с несколькими кафедрами: {len(rows)}")
    print(f"\n{'courseid':<12} {'Кол-во кафедр':>15}")
    print("-"*30)
    for courseid, cnt in rows:
        print(f"{courseid:<12} {cnt:>15}")

    # Показываем кафедры для таких курсов
    print("\nНазвания кафедр, совместно преподающих курсы:")
    for courseid, _ in rows[:5]:
        with conn.cursor() as cur:
            cur.execute("""
                SELECT DISTINCT d.name
                FROM logs l
                JOIN departments d ON l.depart::smallint = d.id
                WHERE l.courseid = %s
            """, (courseid,))
            depts = [r[0] for r in cur.fetchall()]
        print(f"  Курс {courseid}: {', '.join(depts)}")
else:
    print("Курсов с несколькими кафедрами не найдено")


Курсов с несколькими кафедрами: 60

courseid       Кол-во кафедр
------------------------------
78057                     25
79426                      4
75833                      4
72457                      4
72402                      4
72380                      4
72359                      3
84834                      3
72885                      3
87396                      3
72392                      3
88415                      3
75810                      3
75834                      3
75839                      3
75849                      3
72416                      3
71495                      3
72447                      3
72460                      3
83853                      3
71892                      3
71571                      3
82754                      2
82755                      2
84103                      2
84140                      2
84352                      2
86712                      2
87097                      2
87935                      2
88349

## Задание 7: Количество студентов по оценкам (2, 3, 4, 5)

In [14]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            NameR_Level AS namer_level,
            COUNT(DISTINCT userid) AS count
        FROM logs
        WHERE NameR_Level IN ('2','3','4','5')
        GROUP BY NameR_Level
        ORDER BY NameR_Level;
    """)
    rows = cur.fetchall()

print(f"{'namer_level':<15} {'count':>10}")
print("-"*27)
for grade, cnt in rows:
    print(f"{grade:<15} {cnt:>10}")


namer_level          count
---------------------------
2                     1069
3                     1884
4                     3243
5                     3407


## Задание 8: Студент с максимальным количеством логов

In [15]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            userid,
            SUM(s_all) AS total_events
        FROM logs
        GROUP BY userid
        ORDER BY total_events DESC
        LIMIT 1;
    """)
    row = cur.fetchone()

print(f"Самый активный студент:")
print(f"  userid:       {row[0]}")
print(f"  total events: {row[1]:,}")


Самый активный студент:
  userid:       28871
  total events: 10,300


## Задание 9: Среднее количество событий по каждой неделе

In [16]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT
            num_week,
            ROUND(AVG(s_all), 2) AS avg_events
        FROM logs
        GROUP BY num_week
        ORDER BY num_week;
    """)
    rows = cur.fetchall()

print(f"{'Неделя':>8} {'Среднее кол-во событий':>25}")
print("-"*36)
for week, avg in rows:
    print(f"{week:>8} {float(avg):>25.2f}")


  Неделя    Среднее кол-во событий
------------------------------------
       6                     13.80
       7                      9.62
       8                      8.03
       9                      9.39
      10                      8.21
      11                     10.02
      12                      9.38
      13                     10.01
      14                      9.86
      15                     10.35
      16                     10.29
      17                     10.52
      18                      9.67
      19                     11.11
      20                     14.45
      21                     18.50
      22                     22.49
      23                     22.26
      24                     23.01
      25                     18.22
      26                      8.60
      27                      1.25
      28                      0.09
      29                      0.05


## Задание 10: Кафедра с наибольшим числом отличников и двоечников

In [17]:
with conn.cursor() as cur:
    # Отличники (оценка 5)
    cur.execute("""
        SELECT d.name, COUNT(DISTINCT l.userid) AS cnt
        FROM logs l
        JOIN departments d ON l.depart::smallint = d.id
        WHERE l.NameR_Level = '5'
        GROUP BY d.name
        ORDER BY cnt DESC
        LIMIT 1;
    """)
    best = cur.fetchone()

    # Двоечники (оценка 2)
    cur.execute("""
        SELECT d.name, COUNT(DISTINCT l.userid) AS cnt
        FROM logs l
        JOIN departments d ON l.depart::smallint = d.id
        WHERE l.NameR_Level = '2'
        GROUP BY d.name
        ORDER BY cnt DESC
        LIMIT 1;
    """)
    worst = cur.fetchone()

print(f"Кафедра с наибольшим числом отличников (5): {best[0]} ({best[1]} студентов)")
print(f"Кафедра с наибольшим числом двоечников (2): {worst[0]} ({worst[1]} студентов)")


Кафедра с наибольшим числом отличников (5): ДиСО (310 студентов)
Кафедра с наибольшим числом двоечников (2): Эконом. (72 студентов)
